# Session 1: balls, springs and the idea that motion follows forces
**A. Real-world problem.** A protein is a long chain of atoms. Nobody can watch the atoms
directly, so scientists *simulate* them on a computer: they write down the force on every
atom and let Newton's law move the atoms a tiny step at a time. Today we do this for the
smallest possible system, two atoms, using two force laws: a **spring** (a chemical bond)
and the **Lennard-Jones** law (two atoms that attract when apart and repel when squeezed).

**Important honesty note.** Everything in this repo is a *toy model* in made-up "reduced"
units. It teaches how molecular dynamics works; it is not a real protein and not a tool for
any medical or drug decision. General education only.

**Reduced units.** We measure length in sigma, energy in epsilon and mass in m (the atom
mass), so all three equal 1. Time then comes out in tau = sigma * sqrt(m / epsilon).

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
R_MIN = 2 ** (1 / 6)                 # separation where the Lennard-Jones force is zero (sigma)
K_MATCH = 36 * 2 ** (2 / 3)          # spring constant that matches the LJ well at its bottom

## Step 1: a spring and a Lennard-Jones pair
Both are described by a potential energy V(r), where r is the distance between the two atoms.
The force is minus the slope of V:  F(r) = -dV/dr.  Positive F pushes the atoms apart,
negative F pulls them together.

    spring:   V = (1/2) k (r - r0)^2          F = -k (r - r0)
    LJ:       V = 4 [ (1/r)^12 - (1/r)^6 ]    F = 24 [ 2 (1/r)^12 - (1/r)^6 ] / r

In [ ]:
def spring_potential(r, k=K_MATCH, r0=R_MIN):
    return 0.5 * k * (np.asarray(r, dtype=float) - r0) ** 2


def spring_force(r, k=K_MATCH, r0=R_MIN):
    return -k * (np.asarray(r, dtype=float) - r0)


def lj_potential(r):
    s6 = (1.0 / np.asarray(r, dtype=float)) ** 6
    return 4.0 * (s6 * s6 - s6)


def lj_force(r):
    r = np.asarray(r, dtype=float)
    s6 = (1.0 / r) ** 6
    return 24.0 * (2.0 * s6 * s6 - s6) / r

## Step 2: let the force move the atoms
Newton: acceleration = force / mass. For two equal atoms the *distance* r behaves like one
ball of "reduced mass" mu = m/2 pushed by F(r). We step time forward in small steps dt:

    v_new = v + (F / mu) * dt        r_new = r + v_new * dt

In [ ]:
def simulate_pair(force, r_start, steps=2000, dt=0.002, mu=0.5):
    """Distance r(t) of a two-atom pair released at rest. Returns (times, r, v)."""
    t = np.arange(steps + 1) * dt
    r = np.empty(steps + 1)
    v = np.empty(steps + 1)
    r[0], v[0] = r_start, 0.0
    for i in range(steps):
        v[i + 1] = v[i] + force(r[i]) / mu * dt
        r[i + 1] = r[i] + v[i + 1] * dt
    return t, r, v


def measured_period(t, r):
    """Average time between successive maxima of r(t) (simple peak finder)."""
    peaks = [i for i in range(1, len(r) - 1) if r[i] > r[i - 1] and r[i] >= r[i + 1]]
    if len(peaks) < 2:
        return float("nan")
    return float((t[peaks[-1]] - t[peaks[0]]) / (len(peaks) - 1))


def spring_period(k, mu=0.5):
    """Exact period of a spring: 2 pi sqrt(mu / k)."""
    return 2 * np.pi * np.sqrt(mu / k)

## Step 3: look at it
The plots show the two energy curves, the two force curves and the motion. The spring and
the LJ well agree near the bottom (that is how K_MATCH was chosen) but they disagree far
away: a spring keeps pulling harder, LJ lets go.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    r = np.linspace(0.95, 2.5, 300)
    fig, axs = plt.subplots(1, 2, figsize=(9.5, 4))
    axs[0].plot(r, lj_potential(r), label="Lennard-Jones V")
    axs[0].plot(r, spring_potential(r), "--", label="spring V (k = %.1f)" % K_MATCH)
    axs[0].axhline(0, color="gray", lw=0.5)
    axs[0].set_ylim(-1.5, 3)
    axs[0].set_xlabel("distance r (sigma)")
    axs[0].set_ylabel("potential energy V (epsilon)")
    axs[0].set_title("Energy: the well")
    axs[1].plot(r, lj_force(r), label="Lennard-Jones F")
    axs[1].plot(r, spring_force(r), "--", label="spring F")
    axs[1].axhline(0, color="gray", lw=0.5)
    axs[1].set_ylim(-8, 12)
    axs[1].set_xlabel("distance r (sigma)")
    axs[1].set_ylabel("force F (epsilon / sigma)")
    axs[1].set_title("Force = minus the slope of V")
    for a in axs:
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "potentials.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 3.8))
    for name, f in (("spring", spring_force), ("Lennard-Jones", lj_force)):
        t, rr, _ = simulate_pair(f, R_MIN + 0.1)
        ax2.plot(t, rr, label="%s: period %.3f tau" % (name, measured_period(t, rr)))
    ax2.axhline(R_MIN, color="gray", ls=":")
    ax2.set_xlabel("time t (tau)")
    ax2.set_ylabel("distance r (sigma)")
    ax2.set_title("A pair released 0.1 sigma from the bottom of the well")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "pair_motion.png", dpi=120)
    print("LJ minimum at r = %.4f sigma, depth %.3f epsilon" % (R_MIN, lj_potential(R_MIN)))
    print("LJ at r=1.5: V = %.4f, F = %.4f" % (lj_potential(1.5), lj_force(1.5)))
    print("spring constant matching the LJ well: %.2f" % K_MATCH)
    print("exact spring period %.4f tau" % spring_period(K_MATCH))
    t, rr, _ = simulate_pair(spring_force, R_MIN + 0.1)
    print("simulated spring period %.4f tau" % measured_period(t, rr))
    t, rr, _ = simulate_pair(lj_force, R_MIN + 0.1)
    print("simulated LJ period %.4f tau" % measured_period(t, rr))
    t, rr, _ = simulate_pair(lj_force, R_MIN + 0.5, steps=4000)
    print("LJ large swing (0.5 sigma) period %.4f tau" % measured_period(t, rr))

## Where this stops
Two atoms cannot fold. Session 2 makes the stepping rule accurate (velocity Verlet) and adds
a whole chain; Session 3 adds heat (a thermostat) and watches a chain collapse.

## Coding tasks
1. **Easy.** Change the spring constant K_MATCH in `lesson.py` to 20 and 200 and predict, then check, the period of the simulated spring pair.
2. **Stretch.** Write `lj_zero(r)` that returns the Lennard-Jones distance where the energy (not the force) is zero, and check that it equals 1 sigma.
3. **Explain.** Explain in your own words why the force is zero at the bottom of the well but the energy is -1, not 0.

## Assessment
- **A. Conceptual.** A spring pair and a Lennard-Jones pair both oscillate about the same bottom. Why does the spring have a fixed period but the Lennard-Jones pair's period depends on how far you stretch it?
- **B. Numerical.** A bond is modelled by a spring with k = 10 and r0 = 1. Find its energy and force at r = 1.3 (all in reduced units), and the Lennard-Jones energy and force at r = 2.
- **C. Graph and simulation interpretation.** On the energy graph, where is the force zero, and what happens to a pair released at r = 1.6 sigma? Read it from the slope.
- **D. Design and optimisation challenge.** Design: choose a spring constant so that a bond pair oscillates with a period of 0.5 tau, and say how large a time step you would then need to follow the motion with about 20 steps per period.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("LJ force is zero at r = 2^(1/6) = 1.1225", abs(float(L.lj_force(L.R_MIN))) < 1e-9))
    checks.append(("LJ well depth at the minimum is -1 epsilon", abs(float(L.lj_potential(L.R_MIN)) + 1.0) < 1e-9))
    checks.append(("LJ potential is zero at r = 1 sigma", abs(float(L.lj_potential(1.0))) < 1e-12))

    # worked example: LJ at r = 1.5
    v, f = float(L.lj_potential(1.5)), float(L.lj_force(1.5))
    checks.append((f"worked example: LJ at r=1.5 gives V = -0.3203 and F = -1.158 (got {v:.4f}, {f:.3f})",
                   abs(v + 0.3203) < 1e-3 and abs(f + 1.158) < 2e-3))

    # force = minus slope of V (central difference), for both laws
    h = 1e-6
    for name, V, F in (("LJ", L.lj_potential, L.lj_force), ("spring", L.spring_potential, L.spring_force)):
        r = 1.4
        slope = (float(V(r + h)) - float(V(r - h))) / (2 * h)
        checks.append((f"{name}: force equals minus the slope of the potential", abs(float(F(r)) + slope) < 1e-4))

    # hand calculation: spring at r = 1.3 with k = 10, r0 = 1: V = 0.5*10*0.09 = 0.45, F = -3
    checks.append(("hand calculation: spring k=10, r0=1, r=1.3 gives V=0.45 and F=-3.0",
                   abs(float(L.spring_potential(1.3, 10.0, 1.0)) - 0.45) < 1e-9 and abs(float(L.spring_force(1.3, 10.0, 1.0)) + 3.0) < 1e-9))

    # the matched spring has the same curvature as the LJ well
    h = 1e-4
    curv = (float(L.lj_potential(L.R_MIN + h)) - 2 * float(L.lj_potential(L.R_MIN)) + float(L.lj_potential(L.R_MIN - h))) / h ** 2
    checks.append((f"spring constant matches the LJ curvature (57.15, got {curv:.2f})", abs(curv - L.K_MATCH) < 0.05))

    # simulated spring period equals 2 pi sqrt(mu/k)
    t, r, _ = L.simulate_pair(L.spring_force, L.R_MIN + 0.1)
    checks.append(("simulated spring period matches 2 pi sqrt(mu/k) within 1 percent",
                   abs(L.measured_period(t, r) / L.spring_period(L.K_MATCH) - 1) < 0.01))

    # edge case: released exactly at the minimum, nothing moves
    t, r, _ = L.simulate_pair(L.lj_force, L.R_MIN, steps=200)
    checks.append(("edge case: a pair at the bottom of the well stays put", float(np.max(np.abs(r - L.R_MIN))) < 1e-9))
    # limit: far apart the LJ force is tiny and attractive
    checks.append(("limit: at r = 5 sigma the LJ force is attractive and below 0.01",
                   -0.01 < float(L.lj_force(5.0)) < 0))
    # LJ is anharmonic: a big swing takes longer than a small one
    t1, r1, _ = L.simulate_pair(L.lj_force, L.R_MIN + 0.1)
    t2, r2, _ = L.simulate_pair(L.lj_force, L.R_MIN + 0.5, steps=4000)
    checks.append(("LJ period grows with the size of the swing", L.measured_period(t2, r2) > 1.5 * L.measured_period(t1, r1)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
